# 《LangChain MCP项目实践实验》

## 一、实验目的
1. 理解 MCP（Model Context Protocol）的核心概念与工作原理
2. 掌握使用 FastMCP 创建 MCP Server 的方法
3. 掌握通过 LangChain MCP Adapters 将 MCP 工具转换为 LangChain 工具
4. 理解 MCP 配置文件的结构与作用
5. 完成 MCP + LangChain + Agent 的完整项目实践

## 二、实验环境
- 系统：Windows 10
- Python版本：3.10
- 虚拟环境：Miniconda
- 开发工具：VS Code / Jupyter Notebook
- 依赖库：langchain、langchain-core、langchain-mcp-adapters、langgraph、FastMCP、python-dotenv、httpx
- 模型：DeepSeek（deepseek-chat）


## 三、实验原理

### 3.1 MCP介绍

MCP（Model Context Protocol）是一种开放协议，用于标准化应用程序向大语言模型提供上下文的方式。MCP 通过客户端-服务器架构实现，允许开发人员通过 MCP Server 向 AI 应用程序安全地暴露数据、工具和功能。

LangChain 调用 MCP 是可以将 MCP 的工具直接转换为 LangChain 的工具，然后通过预定义的 MCP_Client 实现与外部 MCP 的读写操作。换而言之，就是我们需要改写原先的 client，将原先的 Function calling 调用逻辑修改为 LangChain 调用逻辑。

### 3.2 架构说明

整个项目由三部分组成：

1. **MCP Server**：使用 FastMCP 框架创建，对外暴露工具（如天气查询），通过 SSE 协议传输
2. **MCP 配置文件**：mcp.json 文件，定义 MCP Server 的连接地址和传输方式
3. **LangChain 客户端**：使用 MultiServerMCPClient 连接 MCP Server，自动发现工具，通过 create_react_agent 创建 Agent 调用工具

工作流程：

1. 启动 MCP Server（暴露天气查询工具）
2. LangChain 客户端读取 mcp.json 配置，连接 MCP Server
3. 客户端自动发现 MCP Server 提供的工具
4. 创建 Agent，将 MCP 工具作为 LangChain 工具使用
5. 用户提问 -> Agent 推理 -> 调用 MCP 工具 -> 返回结果

## 四、实验内容

### 4.1 创建MCP Server

使用 FastMCP 框架创建一个天气查询的 MCP Server，通过 SSE 协议对外提供服务。

In [ ]:
%pip install fastmcp==3.2.4

In [ ]:
import json
import os
import httpx
import dotenv
from loguru import logger
from mcp.server.fastmcp import FastMCP

dotenv.load_dotenv()

# 创建 MCP Server（SSE模式）
mcp = FastMCP(
    name="WeatherServerSSE",
    host="0.0.0.0",
    port=8000
)


@mcp.tool()
def get_weather(city: str) -> str:
    """
    查询指定城市的即时天气信息（OpenWeather API）

    参数:
        city: 城市英文名，如 Beijing / Shanghai

    返回:
        JSON字符串（天气数据）
    """
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric",
        "lang": "zh_cn"
    }
    resp = httpx.get(url, params=params, timeout=10)
    data = resp.json()
    logger.info(f"[MCP] {city} 天气查询完成")
    return json.dumps(data, ensure_ascii=False)


if __name__ == "__main__":
    logger.info("MCP Weather Server 启动中...")
    logger.info("SSE地址: http://0.0.0.0:8000/sse")
    mcp.run(transport="sse")

### 4.2 创建MCP配置文件

创建 mcp.json 文件，定义 MCP Server 的连接地址和传输方式。

In [ ]:
{
  "mcpServers": {
    "weather": {
      "url": "http://127.0.0.1:8000/sse",
      "transport": "sse"
    }
  }
}

### 4.3 LangChain客户端

使用 MultiServerMCPClient 连接 MCP Server，自动发现工具，通过 create_react_agent 创建 Agent 实现智能对话。

In [ ]:
%pip install langchainhub==0.1.21

In [ ]:
import asyncio
import json
import os
from dotenv import load_dotenv
from loguru import logger
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain.chat_models import init_chat_model

load_dotenv(override=True)


def load_servers(file_path="mcp.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)["mcpServers"]


async def main():
    # 1. 加载 MCP Server配置
    servers = load_servers()

    # 2. 创建 MCP Client
    client = MultiServerMCPClient(servers)

    # 3. 获取 MCP Tools（自动发现）
    tools = await client.get_tools()
    logger.info(f"已加载工具: {[t.name for t in tools]}")

    # 4. 初始化 LLM
    llm = init_chat_model(
        "deepseek-chat",
        model_provider="deepseek",
        api_key=os.getenv("DEEPSEEK_API_KEY")
    )

    # 5. 创建 Agent
    agent = create_react_agent(llm, tools)

    # 6. CLI循环
    logger.info("MCP Agent 已启动，输入 quit 退出")

    while True:
        query = input("\n你: ").strip()
        if query.lower() == "quit":
            break
        try:
            result = await agent.ainvoke({
                "messages": [("user", query)]
            })
            final_msg = result["messages"][-1].content
            print("\nAI:", final_msg)
        except Exception as e:
            logger.error(f"错误: {e}")


if __name__ == "__main__":
    asyncio.run(main())

## 五、实验总结

1. 理解了 MCP（Model Context Protocol）的核心概念，认识到 MCP 在标准化 AI 应用与外部工具交互中的重要作用
2. 使用 FastMCP 框架创建了天气查询 MCP Server，掌握了 @mcp.tool() 装饰器定义工具和 SSE 传输模式的启动方式
3. 通过 mcp.json 配置文件管理 MCP Server 连接信息，理解了配置化的工程实践
4. 使用 MultiServerMCPClient 连接 MCP Server，实现了 MCP 工具到 LangChain 工具的自动转换
5. 通过 create_react_agent 创建 Agent，完成了 MCP + LangChain 的完整项目实践